# Speech-to-Text Transcription Quality Improvement Pipeline

**Dataset**: LJSpeech-1.1 (single-speaker English, ~24 h)  
**STT Engine**: faster-whisper (Whisper base)  
**Post-processing**: Text normalisation + Vocabulary biasing + LLM correction (Ollama/qwen3.5:4b)  
**Metrics**: WER · CER · MER

---

## Pipeline Overview

```
Audio Files  ──►  faster-whisper (base)  ──►  Baseline Hypothesis
                                                        │
                                          ┌─────────────▼──────────────┐
                                          │  Stage 1: Text Normalise   │
                                          │  Stage 2: Vocab Biasing    │
                                          │  Stage 3: LM Correction    │
                                          └─────────────┬──────────────┘
                                                        │
                                          ┌─────────────▼──────────────┐
                                          │  WER / CER Evaluation      │
                                          └────────────────────────────┘
```

## 0. Setup

In [ ]:
import os, sys, json, re, time, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from faster_whisper import WhisperModel
from jiwer import wer, cer, mer, process_words
from jiwer import Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip

# Add src/ to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

LJSPEECH_DIR  = '/home/muthuajay/Documents/Datasets/LJSpeech-1.1'
LJSPEECH_WAVS = os.path.join(LJSPEECH_DIR, 'wavs')
META_CSV      = os.path.join(LJSPEECH_DIR, 'metadata.csv')
OUTPUTS_DIR   = str(Path.cwd().parent / 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

TRANSFORM = Compose([ToLowerCase(), RemovePunctuation(), RemoveMultipleSpaces(), Strip()])

print('Setup complete ✓')

---
## 1. Data Collection

In [ ]:
# Load full LJSpeech metadata
meta = pd.read_csv(META_CSV, sep='|', header=None,
                   names=['file_id', 'transcript', 'normalized_transcript'])
print(f'Total LJSpeech utterances: {len(meta):,}')

# Estimate total audio duration (LJSpeech avg ~4.3s per clip)
AVG_CLIP_SEC = 4.3
total_hours = len(meta) * AVG_CLIP_SEC / 3600
print(f'Estimated total audio: ~{total_hours:.1f} hours')

# Sample 200 utterances for evaluation
NUM_SAMPLES = 200
sampled = meta.sample(NUM_SAMPLES, random_state=42).reset_index(drop=True)
sampled['audio_path'] = sampled['file_id'].apply(
    lambda fid: os.path.join(LJSPEECH_WAVS, f'{fid}.wav')
)
sampled = sampled[sampled['audio_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'\nSampled: {len(sampled)} utterances')

In [ ]:
# Dataset statistics
word_counts = sampled['transcript'].str.split().str.len()
char_counts = sampled['transcript'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(word_counts, bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Words per Utterance')
axes[0].set_ylabel('Count')
axes[0].set_title('Word Count Distribution')

axes[1].hist(char_counts, bins=20, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Characters per Utterance')
axes[1].set_ylabel('Count')
axes[1].set_title('Character Count Distribution')

plt.suptitle('LJSpeech Sample Statistics', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'dataset_stats.png'), dpi=150)
plt.show()

print(f'Avg words/utt : {word_counts.mean():.1f}  (std={word_counts.std():.1f})')
print(f'Avg chars/utt : {char_counts.mean():.1f}')
sampled.head()

---
## 2. Baseline Transcription

We use **faster-whisper** with the `base` model — a highly efficient CTranslate2 reimplementation of OpenAI Whisper.

| Model | Params | VRAM | Rel. Speed |
|-------|--------|------|------------|
| tiny  | 39 M   | ~1 GB | ~10× |
| **base**  | **74 M** | **~1 GB** | **~7×** |
| small | 244 M  | ~2 GB | ~4× |
| medium| 769 M  | ~5 GB | ~2× |

In [ ]:
# Load faster-whisper base model
model = WhisperModel('base', compute_type='float16')
print('Model loaded ✓')

def transcribe(audio_path: str) -> str:
    segments, _ = model.transcribe(
        audio_path, language='en',
        beam_size=5, vad_filter=True
    )
    return ' '.join(seg.text.strip() for seg in segments).strip()

In [ ]:
baseline_csv = os.path.join(OUTPUTS_DIR, 'baseline_results.csv')

if os.path.exists(baseline_csv):
    print(f'Loading cached baseline results from {baseline_csv}')
    results_df = pd.read_csv(baseline_csv)
    # Merge with sampled to keep all columns
    results_df = sampled.merge(results_df[['file_id','hypothesis','runtime_s']],
                                on='file_id', how='left')
else:
    hypotheses, runtimes = [], []
    for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc='Transcribing'):
        t0 = time.perf_counter()
        hyp = transcribe(row['audio_path'])
        runtimes.append(round(time.perf_counter() - t0, 3))
        hypotheses.append(hyp)

    results_df = sampled.copy()
    results_df['hypothesis'] = hypotheses
    results_df['runtime_s']  = runtimes
    results_df.to_csv(baseline_csv, index=False)
    print(f'Saved → {baseline_csv}')

print(f'\nAvg inference time: {results_df["runtime_s"].mean():.3f}s/utt')
results_df[['file_id','transcript','hypothesis','runtime_s']].head()

---
## 3. Evaluation – WER, CER, MER

In [ ]:
def compute_metrics(refs, hyps, label=''):
    refs_n = [TRANSFORM(r) for r in refs]
    hyps_n = [TRANSFORM(h) for h in hyps]
    metrics = {
        'WER (%)': round(wer(refs_n, hyps_n) * 100, 2),
        'CER (%)': round(cer(refs_n, hyps_n) * 100, 2),
        'MER (%)': round(mer(refs_n, hyps_n) * 100, 2),
    }
    if label:
        print(f'\n=== {label} ===')
        for k, v in metrics.items():
            print(f'  {k}: {v:.2f}%')
    return metrics

baseline_metrics = compute_metrics(
    results_df['transcript'].tolist(),
    results_df['hypothesis'].tolist(),
    label='Baseline (whisper-base)'
)

In [ ]:
# Per-utterance WER
utt_wers = []
for _, row in results_df.iterrows():
    ref = TRANSFORM(str(row['transcript']))
    hyp = TRANSFORM(str(row['hypothesis']))
    utt_wers.append(wer(ref, hyp) * 100)
results_df['utt_wer'] = utt_wers

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
axes[0].hist(utt_wers, bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(utt_wers), color='red', linestyle='--',
                label=f'Mean={np.mean(utt_wers):.1f}%')
axes[0].set_xlabel('WER (%)'); axes[0].set_ylabel('Utterances')
axes[0].set_title('Per-Utterance WER Distribution')
axes[0].legend()

# Buckets
buckets = {'0%': 0, '1–10%': 0, '11–30%': 0, '31–60%': 0, '>60%': 0}
for w in utt_wers:
    if   w == 0:   buckets['0%']    += 1
    elif w <= 10:  buckets['1–10%'] += 1
    elif w <= 30:  buckets['11–30%']+= 1
    elif w <= 60:  buckets['31–60%']+= 1
    else:          buckets['>60%']  += 1

bars = axes[1].bar(buckets.keys(), buckets.values(),
                   color=sns.color_palette('Blues_d', 5), edgecolor='white')
axes[1].bar_label(bars)
axes[1].set_xlabel('WER Bucket'); axes[1].set_ylabel('Utterances')
axes[1].set_title('WER Bucket Distribution')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'baseline_wer_distribution.png'), dpi=150)
plt.show()

print(f'Perfect (WER=0): {buckets["0%"]} / {len(utt_wers)} ({buckets["0%"]/len(utt_wers)*100:.1f}%)')

In [ ]:
# Show example predictions
print('=== Best 5 Utterances (lowest WER) ===')
for _, row in results_df.nsmallest(5, 'utt_wer').iterrows():
    print(f'  WER={row["utt_wer"]:5.1f}%  REF: {row["transcript"][:70]}')
    print(f'               HYP: {row["hypothesis"][:70]}')
    print()

print('=== Worst 5 Utterances (highest WER) ===')
for _, row in results_df.nlargest(5, 'utt_wer').iterrows():
    print(f'  WER={row["utt_wer"]:5.1f}%  REF: {row["transcript"][:70]}')
    print(f'               HYP: {row["hypothesis"][:70]}')
    print()

---
## 4. Error Analysis

In [ ]:
from collections import defaultdict

substitutions = Counter()
deletions     = Counter()
insertions    = Counter()
total_s = total_i = total_d = total_ref = 0

for _, row in results_df.iterrows():
    ref = TRANSFORM(str(row['transcript']))
    hyp = TRANSFORM(str(row['hypothesis']))
    result = process_words(ref, hyp)

    for chunk in result.alignments[0]:
        if chunk.type == 'substitute':
            rw = result.references[0][chunk.ref_start_idx:chunk.ref_end_idx]
            hw = result.hypotheses[0][chunk.hyp_start_idx:chunk.hyp_end_idx]
            for r, h in zip(rw, hw):
                substitutions[(r, h)] += 1
            total_s += len(rw)
        elif chunk.type == 'delete':
            for w in result.references[0][chunk.ref_start_idx:chunk.ref_end_idx]:
                deletions[w] += 1
            total_d += chunk.ref_end_idx - chunk.ref_start_idx
        elif chunk.type == 'insert':
            for w in result.hypotheses[0][chunk.hyp_start_idx:chunk.hyp_end_idx]:
                insertions[w] += 1
            total_i += chunk.hyp_end_idx - chunk.hyp_start_idx

    total_ref += len(result.references[0])

total_errors = total_s + total_d + total_i
print(f'Total reference words : {total_ref}')
print(f'Total errors          : {total_errors}')
print(f'  Substitutions : {total_s:>5}  ({total_s/total_errors*100:.1f}%)')
print(f'  Deletions     : {total_d:>5}  ({total_d/total_errors*100:.1f}%)')
print(f'  Insertions    : {total_i:>5}  ({total_i/total_errors*100:.1f}%)')

In [ ]:
# Error operation pie chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels = ['Substitutions', 'Deletions', 'Insertions']
sizes  = [total_s, total_d, total_i]
colors = ['#4C72B0', '#DD8452', '#55A868']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('Error Operation Breakdown')

# Top substitutions
top_subs = substitutions.most_common(12)
sub_labels = [f"'{r}' → '{h}'" for (r, h), _ in top_subs]
sub_counts = [c for _, c in top_subs]
bars = axes[1].barh(sub_labels[::-1], sub_counts[::-1], color='#DD8452', edgecolor='white')
axes[1].bar_label(bars, padding=3)
axes[1].set_xlabel('Count')
axes[1].set_title('Top 12 Substitution Pairs')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'error_analysis.png'), dpi=150)
plt.show()

In [ ]:
# Categorise errors
print('=== Top 15 Substitution Pairs ===')
for (ref_w, hyp_w), cnt in substitutions.most_common(15):
    print(f"  '{ref_w}' → '{hyp_w}'  (×{cnt})")

print('\n=== Top 10 Deleted Words ===')
for w, cnt in deletions.most_common(10):
    print(f"  '{w}'  (×{cnt})")

print('\n=== Top 10 Inserted Words ===')
for w, cnt in insertions.most_common(10):
    print(f"  '{w}'  (×{cnt})")

In [ ]:
# Error pattern categorisation
categories = defaultdict(list)
for (r, h), cnt in substitutions.most_common(50):
    if re.fullmatch(r'\d+(\.*\d+)?', r) or re.fullmatch(r'\d+(\.*\d+)?', h):
        categories['number_mismatch'].append((r, h, cnt))
    elif any(c.isupper() for c in r+h):
        categories['proper_noun_candidate'].append((r, h, cnt))
    elif len(r) <= 3 or len(h) <= 3:
        categories['short_function_word'].append((r, h, cnt))
    else:
        categories['content_word'].append((r, h, cnt))

print('\n=== Error Categories ===')
for cat, items in categories.items():
    total = sum(c for _, _, c in items)
    examples = ', '.join(f"'{r}'→'{h}'" for r, h, _ in items[:3])
    print(f'  {cat:<30}: {len(items):>3} pairs, {total:>4} total  e.g. {examples}')

---
## 5. Improvement

Three complementary strategies:

| # | Strategy | What it fixes |
|---|----------|---------------|
| 1 | **Text Normalisation** | Trailing ellipsis, dash noise, double spaces |
| 2 | **Vocabulary Biasing** | Systematic mis-recognitions from error analysis |
| 3 | **LM Post-correction** | Grammar, missing function words, context-dependent errors |

In [ ]:
# ── Strategy 1: Text normalisation ──────────────────────────────────────
NORM_RULES = [
    (r'\s*\.\.\.\s*$',  ''),
    (r'\s*-\s*$',       ''),
    (r'  +',            ' '),
    (r',\s+(and|but|or|so)\b', r' \1'),
]

def text_normalise(text):
    for pat, repl in NORM_RULES:
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    return text.strip()

# ── Strategy 2: Vocabulary biasing ───────────────────────────────────────
# Build auto-corrections from top substitutions (ref → hyp means hyp appears
# in the hypothesis when ref was said; so we correct hyp → ref)
auto_corrections = {}
for (ref_w, hyp_w), cnt in substitutions.most_common(50):
    if cnt >= 2 and hyp_w != ref_w:
        auto_corrections[f' {hyp_w} '] = f' {ref_w} '

def vocab_bias_correct(text):
    padded = f' {text} '
    for wrong, right in auto_corrections.items():
        padded = padded.replace(wrong, right)
    return padded.strip()

print(f'Auto-correction rules loaded: {len(auto_corrections)}')

# ── Strategy 3: LM post-correction ───────────────────────────────────────
from ollama import chat

LM_SYSTEM = """You are an expert speech-to-text post-correction system.
Fix transcription errors while preserving the original meaning exactly.
Rules:
- Correct grammar mistakes and missing function words
- Fix proper nouns and technical terms if obvious from context
- Do NOT add new information or change meaning
- Return ONLY the corrected sentence, nothing else"""

def lm_correct(text):
    try:
        resp = chat(
            model='qwen3.5:4b',
            messages=[
                {'role': 'system', 'content': LM_SYSTEM},
                {'role': 'user', 'content': f'Fix: {text}'},
            ],
            options={'temperature': 0.0, 'num_predict': 200},
        )
        result = resp['message']['content'].strip()
        result = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()
        return result or text
    except Exception as e:
        return text

# Quick smoke test
test = 'We trained the transformer model using Pytosh and fine tune data on a custom data set.'
print(f'LM test input : {test}')
print(f'LM test output: {lm_correct(test)}')

In [ ]:
improved_csv = os.path.join(OUTPUTS_DIR, 'improved_results.csv')

if os.path.exists(improved_csv):
    print(f'Loading cached improved results …')
    improved_df = pd.read_csv(improved_csv)
else:
    after_norm, after_vocab, after_lm = [], [], []

    for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc='Improving'):
        s1 = text_normalise(str(row['hypothesis']))
        s2 = vocab_bias_correct(s1)
        s3 = lm_correct(s2)
        after_norm.append(s1)
        after_vocab.append(s2)
        after_lm.append(s3)

    improved_df = results_df.copy()
    improved_df['hypothesis_norm']  = after_norm
    improved_df['hypothesis_vocab'] = after_vocab
    improved_df['hypothesis']       = after_lm
    improved_df.to_csv(improved_csv, index=False)
    print(f'Saved → {improved_csv}')

improved_df.head(3)

In [ ]:
# Show sample improvements
sample_idx = improved_df.sample(5, random_state=7).index
for i in sample_idx:
    row = improved_df.loc[i]
    print(f'REF  : {row["transcript"]}')
    print(f'BASE : {row["hypothesis_norm"]}')
    print(f'IMPR : {row["hypothesis"]}')
    print()

---
## 6. Re-evaluation

In [ ]:
refs = improved_df['transcript'].tolist()

stages = {
    'Baseline':      improved_df['hypothesis_norm'].tolist() if 'hypothesis_norm' in improved_df.columns
                     else results_df['hypothesis'].tolist(),
    'After Norm.':   improved_df.get('hypothesis_norm', pd.Series()).tolist(),
    'After Vocab':   improved_df.get('hypothesis_vocab', pd.Series()).tolist(),
    'After LM':      improved_df['hypothesis'].tolist(),
}

rows = []
for label, hyps in stages.items():
    if hyps and any(str(h) != 'nan' for h in hyps):
        m = compute_metrics(refs, [str(h) for h in hyps])
        rows.append({'Stage': label, **m})

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, metric in zip(axes, ['WER (%)', 'CER (%)']):
    bars = ax.bar(comparison_df['Stage'], comparison_df[metric],
                  color=colors[:len(comparison_df)], edgecolor='white', width=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Processing Stage')
    ax.set_ylim(0, comparison_df[metric].max() * 1.3)
    ax.bar_label(bars, fmt='%.2f%%', padding=3)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('STT Improvement: Stage-by-Stage Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'improvement_comparison.png'), dpi=150)
plt.show()

# Delta summary
base_wer   = comparison_df.loc[comparison_df['Stage']=='Baseline', 'WER (%)'].values[0]
final_wer  = comparison_df.loc[comparison_df['Stage']=='After LM',  'WER (%)'].values[0]
delta      = base_wer - final_wer
rel_improv = delta / base_wer * 100
print(f'\nWER: {base_wer:.2f}% → {final_wer:.2f}%  (Δ {delta:.2f}%, {rel_improv:.1f}% relative improvement)')

In [ ]:
# Top improved utterances
improved_df['base_wer_val'] = [wer(TRANSFORM(r), TRANSFORM(h)) * 100
    for r, h in zip(improved_df['transcript'], improved_df.get('hypothesis_norm', improved_df['hypothesis']))]
improved_df['final_wer_val'] = [wer(TRANSFORM(r), TRANSFORM(h)) * 100
    for r, h in zip(improved_df['transcript'], improved_df['hypothesis'])]
improved_df['wer_delta'] = improved_df['base_wer_val'] - improved_df['final_wer_val']

print('=== Top 5 Most Improved Utterances ===')
for _, row in improved_df.nlargest(5, 'wer_delta').iterrows():
    print(f'REF  : {row["transcript"][:80]}')
    b = row.get('hypothesis_norm', row['hypothesis'])
    print(f'BASE : {str(b)[:80]}')
    print(f'IMPR : {row["hypothesis"][:80]}')
    print(f'WER  : {row["base_wer_val"]:.1f}% → {row["final_wer_val"]:.1f}%  (Δ {row["wer_delta"]:.1f}%)')
    print()

---
## 7. Streaming / Live STT Demo

In [ ]:
"""
Streaming Demo – simulates real-time transcription by feeding a sample
audio file in 4-second chunks (matching a live mic scenario).

To run with the microphone instead, execute:
    python src/07_streaming_demo.py               # chunk mode
    python src/07_streaming_demo.py --mode vad    # voice-activity mode
"""
import tempfile, wave
import sounddevice as sd
from scipy.io import wavfile

SAMPLERATE = 16000
CHUNK_SEC  = 4.0

def transcribe_chunk_arr(model, audio_arr, sr=16000):
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    audio_i16 = (audio_arr * 32767).astype(np.int16)
    with wave.open(tmp_path, 'wb') as wf:
        wf.setnchannels(1); wf.setsampwidth(2)
        wf.setframerate(sr); wf.writeframes(audio_i16.tobytes())
    segs, _ = model.transcribe(tmp_path, language='en', beam_size=3, vad_filter=True)
    text = ' '.join(s.text.strip() for s in segs).strip()
    os.unlink(tmp_path)
    return text

# File streaming demo using the first sample file
demo_file = results_df.iloc[0]['audio_path']
sr, data  = wavfile.read(demo_file)
if data.ndim > 1: data = data.mean(axis=1)
if data.dtype != np.float32:
    data = data.astype(np.float32) / np.iinfo(data.dtype).max

chunk_sz = int(CHUNK_SEC * sr)
print(f'File: {os.path.basename(demo_file)}  ({len(data)/sr:.1f}s)')
print(f'REF : {results_df.iloc[0]["transcript"]}')
print()

parts = []
for i in range(0, len(data), chunk_sz):
    chunk = data[i:i+chunk_sz]
    if len(chunk) < sr * 0.3: break
    t0   = time.perf_counter()
    text = transcribe_chunk_arr(model, chunk, sr)
    lat  = time.perf_counter() - t0
    if text:
        parts.append(text)
        print(f'[chunk {i//chunk_sz+1}] {text}  ({lat:.2f}s)')

print(f'\nFull transcript: {" ".join(parts)}')

In [ ]:
# Live microphone transcription (run this cell to test with your mic)
# Uncomment to enable:

# RECORD_SEC = 5
# print(f'Recording for {RECORD_SEC}s … speak now!')
# audio = sd.rec(int(RECORD_SEC * SAMPLERATE), samplerate=SAMPLERATE,
#                channels=1, dtype='float32')
# sd.wait()
# audio = audio.flatten()
# text = transcribe_chunk_arr(model, audio, SAMPLERATE)
# print(f'Transcript: {text}')

---
## 8. System Design

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│                      STT Quality Improvement Pipeline                            │
├──────────────────────────────────────────────────────────────────────────────────┤
│                                                                                  │
│  ┌─────────┐     ┌──────────────────┐     ┌──────────────────────────────────┐  │
│  │  Audio  │────►│  faster-whisper  │────►│       Post-Processing Stack      │  │
│  │  Input  │     │  (Whisper base)  │     │                                  │  │
│  │         │     │                  │     │  1. Text Normalisation            │  │
│  │ • File  │     │  • CTranslate2   │     │     regex rules, artifact strip  │  │
│  │ • Mic   │     │  • VAD filter    │     │                                  │  │
│  │ • Stream│     │  • beam_size=5   │     │  2. Vocabulary Biasing            │  │
│  └─────────┘     └──────────────────┘     │     correction dict from error  │  │
│                                            │     analysis                    │  │
│                                            │                                  │  │
│                                            │  3. LLM Post-Correction          │  │
│                                            │     Ollama / qwen3.5:4b          │  │
│                                            │     grammar + context fix        │  │
│                                            └──────────────┬───────────────────┘  │
│                                                           │                       │
│                    ┌──────────────────────────────────────▼──────────────────┐   │
│                    │               Evaluation Layer                          │   │
│                    │   WER · CER · MER  |  per-utt analysis  |  comparison  │   │
│                    └─────────────────────────────────────────────────────────┘   │
└──────────────────────────────────────────────────────────────────────────────────┘
```

### Component Details

| Component | Technology | Role |
|-----------|-----------|------|
| Audio capture | `sounddevice` + `scipy` | Mic/file input at 16 kHz mono |
| ASR engine | `faster-whisper` (base) | CTranslate2 Whisper reimplementation |
| VAD | Built-in Silero VAD | Filters non-speech frames |
| Text normalisation | `re` rules | Remove artifacts, fix punctuation |
| Vocab biasing | Frequency-based dict | Corrects systematic mis-recognitions |
| LM correction | Ollama `qwen3.5:4b` | Context-aware grammar fix |
| Evaluation | `jiwer` | WER / CER / MER computation |

### Key Design Decisions

1. **faster-whisper over plain Whisper**: 4–5× faster on CPU/GPU, lower memory, drop-in replacement.
2. **Three-stage post-processing**: Ordered from cheapest (regex) to most expensive (LLM), allowing the pipeline to be run with partial stages.
3. **Error-analysis-driven vocabulary biasing**: Corrections are derived automatically from the substitution frequency table, making them data-driven and reproducible.
4. **Local LLM (Ollama)**: No API keys, no internet required, privacy-safe.

In [ ]:
# Final summary table
print('=' * 60)
print('           FINAL RESULTS SUMMARY')
print('=' * 60)
print(comparison_df.to_string(index=False))
print('=' * 60)
print(f'\nRelative WER improvement: {rel_improv:.1f}%')
print(f'Absolute WER reduction  : {delta:.2f} pp')